# Comparación de modelos sin y con lags FIRMS

Comparamos ambas variantes con el mismo target y split temporal. El test 2025 queda reservado.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## 1. Carga y validación de los datasets

In [ ]:
datos_sin_lags_firms = pd.read_parquet("dataset_supervisado_sin_lags_firms.parquet")
datos_con_lags_firms = pd.read_parquet("dataset_supervisado_con_lags_firms.parquet")

#Ordenamos ambos datasets para comparar exactamente las mismas observaciones
columnas_clave = ["departamento", "fecha_inicio_semana"]
datos_sin_lags_firms = datos_sin_lags_firms.sort_values(columnas_clave).reset_index(drop=True)
datos_con_lags_firms = datos_con_lags_firms.sort_values(columnas_clave).reset_index(drop=True)

assert datos_sin_lags_firms.shape[0] == datos_con_lags_firms.shape[0] == 7923
assert datos_sin_lags_firms[columnas_clave].equals(datos_con_lags_firms[columnas_clave])
assert datos_sin_lags_firms["nivel_actividad_firms"].equals(datos_con_lags_firms["nivel_actividad_firms"])
assert datos_sin_lags_firms[columnas_clave].duplicated().sum() == 0
assert datos_con_lags_firms[columnas_clave].duplicated().sum() == 0

print("SIN lags FIRMS:", datos_sin_lags_firms.shape)
print("CON lags FIRMS:", datos_con_lags_firms.shape)
print("Período:", datos_sin_lags_firms["fecha_inicio_semana"].min(), "a", datos_sin_lags_firms["fecha_inicio_semana"].max())
print("Mismas claves, target y ausencia de duplicados: Sí")

## 2. Features explícitas

In [ ]:
features_sin_lags_firms = [
    # Semana anterior
    "precipitation_sum_weekly_lag1",
    "temperature_2m_max_mean_weekly_lag1",
    "temperature_2m_min_mean_weekly_lag1",
    "relative_humidity_2m_min_mean_weekly_lag1",
    "wind_speed_10m_max_mean_weekly_lag1",
    "et0_fao_evapotranspiration_sum_weekly_lag1",

    # 2 semanas anteriores
    "precipitation_sum_weekly_prev_2w_sum",
    "temperature_2m_max_mean_weekly_prev_2w_mean",
    "temperature_2m_min_mean_weekly_prev_2w_mean",
    "relative_humidity_2m_min_mean_weekly_prev_2w_mean",
    "wind_speed_10m_max_mean_weekly_prev_2w_mean",
    "et0_fao_evapotranspiration_sum_weekly_prev_2w_sum",

    # 4 semanas anteriores
    "precipitation_sum_weekly_prev_4w_sum",
    "temperature_2m_max_mean_weekly_prev_4w_mean",
    "temperature_2m_min_mean_weekly_prev_4w_mean",
    "relative_humidity_2m_min_mean_weekly_prev_4w_mean",
    "wind_speed_10m_max_mean_weekly_prev_4w_mean",
    "et0_fao_evapotranspiration_sum_weekly_prev_4w_sum"
]

features_firms = [
    "cantidad_detecciones_hist_lag1",
    "detecciones_prev_2w_sum",
    "detecciones_prev_4w_sum",
    "semanas_con_deteccion_prev_4w"
]

features_con_lags_firms = features_sin_lags_firms + features_firms

print("Features SIN lags FIRMS:", len(features_sin_lags_firms))
print("Features CON lags FIRMS:", len(features_con_lags_firms))

## 3. Split temporal común

In [ ]:
#Usamos las mismas filas para train, validation y test
train_sin = datos_sin_lags_firms[datos_sin_lags_firms["anio"] <= 2022].copy()
validation_sin = datos_sin_lags_firms[datos_sin_lags_firms["anio"].between(2023, 2024)].copy()
test_sin = datos_sin_lags_firms[datos_sin_lags_firms["anio"] == 2025].copy()

train_con = datos_con_lags_firms[datos_con_lags_firms["anio"] <= 2022].copy()
validation_con = datos_con_lags_firms[datos_con_lags_firms["anio"].between(2023, 2024)].copy()
test_con = datos_con_lags_firms[datos_con_lags_firms["anio"] == 2025].copy()

for sin_lags, con_lags in [(train_sin, train_con), (validation_sin, validation_con), (test_sin, test_con)]:
    assert sin_lags[columnas_clave].equals(con_lags[columnas_clave])
    assert sin_lags["nivel_actividad_firms"].equals(con_lags["nivel_actividad_firms"])

print("TRAIN:", train_sin.shape[0])
print("VALIDATION:", validation_sin.shape[0])
print("TEST reservado:", test_sin.shape[0])

In [ ]:
target = "nivel_actividad_firms"
orden_clases = ["Sin detección", "Bajo", "Moderado", "Alto"]

X_train_sin = train_sin[features_sin_lags_firms]
X_val_sin = validation_sin[features_sin_lags_firms]
X_test_sin = test_sin[features_sin_lags_firms]

X_train_con = train_con[features_con_lags_firms]
X_val_con = validation_con[features_con_lags_firms]
X_test_con = test_con[features_con_lags_firms]

y_train = train_sin[target]
y_val = validation_sin[target]
y_test = test_sin[target]

for X in [X_train_sin, X_val_sin, X_test_sin, X_train_con, X_val_con, X_test_con]:
    assert X.isnull().sum().sum() == 0

print("Nulos en las features utilizadas: 0")
print("TEST preparado pero no evaluado")

In [ ]:
#Usamos la misma función para evaluar los tres modelos en validation
resultados_modelos = []
recall_modelos = {}

def evaluar_en_validation(variante, modelo, y_real, y_pred):
    accuracy = accuracy_score(y_real, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_real, y_pred)
    macro_f1 = f1_score(y_real, y_pred, average="macro")
    reporte = classification_report(y_real, y_pred, labels=orden_clases, output_dict=True, zero_division=0)

    resultados_modelos.append({"Variante": variante, "Modelo": modelo, "Accuracy": accuracy, "Balanced Accuracy": balanced_accuracy, "Macro F1": macro_f1})
    recall_modelos[variante] = {clase: reporte[clase]["recall"] for clase in orden_clases}

    print(variante)
    print("Accuracy:", round(accuracy, 4))
    print("Balanced Accuracy:", round(balanced_accuracy, 4))
    print("Macro F1:", round(macro_f1, 4))
    print(classification_report(y_real, y_pred, labels=orden_clases, zero_division=0))

    matriz = confusion_matrix(y_real, y_pred, labels=orden_clases)
    ConfusionMatrixDisplay(matriz, display_labels=orden_clases).plot()
    plt.title("Matriz de confusión - " + variante)
    plt.xticks(rotation=25)
    plt.show()

## 4. Línea base única

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_sin, y_train)
y_pred_baseline = baseline.predict(X_val_sin)

evaluar_en_validation("Baseline", "DummyClassifier", y_val, y_pred_baseline)

## 5. Random Forest sin lags FIRMS

In [ ]:
modelo_sin_lags_firms = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

modelo_sin_lags_firms.fit(X_train_sin, y_train)
y_pred_val_sin = modelo_sin_lags_firms.predict(X_val_sin)

evaluar_en_validation("Sin lags FIRMS", "Random Forest", y_val, y_pred_val_sin)

## 6. Random Forest con lags FIRMS

In [ ]:
modelo_con_lags_firms = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

modelo_con_lags_firms.fit(X_train_con, y_train)
y_pred_val_con = modelo_con_lags_firms.predict(X_val_con)

evaluar_en_validation("Con lags FIRMS", "Random Forest", y_val, y_pred_val_con)

## 7. Comparación final en validation

In [ ]:
comparacion_modelos = pd.DataFrame(resultados_modelos)
comparacion_modelos[["Accuracy", "Balanced Accuracy", "Macro F1"]] = comparacion_modelos[["Accuracy", "Balanced Accuracy", "Macro F1"]].round(4)
comparacion_modelos

In [ ]:
comparacion_recall = pd.DataFrame(recall_modelos).T[orden_clases].round(4)
comparacion_recall.index.name = "Variante"
comparacion_recall

## Test reservado

`X_test_sin`, `X_test_con` e `y_test` quedan preparados. No se realizan predicciones ni métricas sobre 2025.